In [ ]:
# Imports
import cProfile
import pstats
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad, outputs, errors
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad
import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
# allow jax print statements to show up in the notebook
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

# Settings
star_position = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')
star_ref = star_position.skyoffset_frame()
distance = 293 #parsecs
v_lsr = 7.5 #km/s
save_folder = "sting_results"

# Files
cubefile = 'test_data/IRAS2A/D2CO_streamer_cluster_data.fits'
file_Tpeak = 'test_data/IRAS2A/D2CO_streamer_cluster_tpeak.fits'

# some constants TODO: these should be imported from elsewhere
G = 6.67430e-11 * (1e-3)**2 * (1.988416e30) / (1.4959787e11) # in au (km/s)^2 * Msol^-1
au_in_km = 1.4959787e8 #km


## 1. Prepare 1D streamer emission from cube

The cube should contain only streamer emission. See 'how do I isolate my streamer emission?'.

In [ ]:
# Set:
n_points = 10 # the number of points we want to reduce the data to
# -----------------------------------------------------------------

# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)
print('Cube spectral axis limits:', cube.spectral_axis.min(), cube.spectral_axis.max())

# Optional: trim cube to smaller region that just contains the streamer.
# Set any of theese to None to not apply that limit.
# ----------------------------------------------------------------
vmin = 6 * u.km/u.s             # e.g. 6 * u.km/u.s
vmax = 8 * u.km/u.s             # e.g. 8 * u.km/u.s  
xmin = -5 * u.arcsec            # e.g. -5 * u.arcsec
xmax = 5 * u.arcsec             # e.g. 5 * u.arcsec 
ymin = -12 * u.arcsec           # e.g. -12 * u.arcsec
ymax = 0.5 * u.arcsec           # e.g. 0.5 * u.arcsec
rms_thresh = 4
streamer_cube = cube
# ----------------------------------------------------------------

# extract streamer subcube with the limits
if (vmin is not None) and (vmax is not None):
    print("trimming cube to given velocity limits")
    streamer_cube = streamer_cube.spectral_slab(vmin, vmax)
if (xmin is not None) and (xmax is not None) and (ymin is not None) and (ymax is not None):
    print("trimming cube to given spatial limits")
    celestial_wcs = streamer_cube.wcs.celestial
    ny, nx = streamer_cube.shape[1], streamer_cube.shape[2]
    # reference sky coord corresponding to the reference pixel in the WCS
    ref_ra, ref_dec = celestial_wcs.wcs.crval
    ref_coord = SkyCoord(ref_ra*u.deg, ref_dec*u.deg, frame=FK5)
    # convert the limits from offsets to sky coords
    corner1 = SkyCoord(ref_coord.ra + xmin, ref_coord.dec + ymin, frame=FK5) #'bottom left'
    corner2 = SkyCoord(ref_coord.ra + xmax, ref_coord.dec + ymax, frame=FK5) #'top right'
    x1, y1 = celestial_wcs.world_to_pixel(corner1)
    x2, y2 = celestial_wcs.world_to_pixel(corner2)
    xmin_pix = max(0, int(np.floor(min(x1, x2))))
    xmax_pix = min(nx, int(np.ceil(max(x1, x2))))
    ymin_pix = max(0, int(np.floor(min(y1, y2))))
    ymax_pix = min(ny, int(np.ceil(max(y1, y2))))
    streamer_cube = streamer_cube[:, ymin_pix:ymax_pix, xmin_pix:xmax_pix]
if rms_thresh is not None:
    rms_estimate = streamer_cube.mad_std()
    streamer_cube = streamer_cube.with_mask(streamer_cube > rms_thresh*rms_estimate) 

print('Trimmed/filtered streamer cube spectral axis limits:', streamer_cube.spectral_axis.min(), streamer_cube.spectral_axis.max())


# Reduce the cube to a 1D streamline
pc_coords, pc_means, pc_stds = extract_streamline.reduce_to_1D(streamer_cube, star_position, n_elements=n_points)
# get metric boundaries for later plotting
partitions = extract_streamline.get_metric_partitions(pc_coords, n_elements=n_points)
metric_boundaries, trace = extract_streamline.sample_metric_boundaries(pc_coords, partitions)


# Give data stuff nice names for later
ra_data, dec_data, v_data = pc_means
ra_sigma, dec_sigma, v_sigma = pc_stds
data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

# plot the extracted morphology
outputs.plot_morphology(
    ra_data=ra_data,
    dec_data=dec_data,
    ra_sigma=ra_sigma,
    dec_sigma=dec_sigma,
    pc_coords=pc_coords,
    show=True,
    save_folder=None,
    metric_boundaries=metric_boundaries,
    title="Extracted Streamer Morphology",
)


## 2. Initial Guess

First set your input params in the format required (separate params to optimise and params to stay fixed). Use units.

In [ ]:
# Parameters to optimize
initial_opt_params = {
    'r0': 1500.0 * u.au,  # au
    'theta0': 40.0 * u.deg,  # degrees
    'phi0': 100.0 * u.deg,  # degrees
    # 'rc': 0.1*1500,  # au
    'omega': 5e-13,  # 1/s
    'v_r0': 0.1 * u.km / u.s, # km/s
    # 'inc': -45.0 * u.deg,  # degrees
    # 'pa': 194.0 * u.deg,  # degrees
    # 'mass': 4.0 * u.Msun,  # solar masses
}

# IRAS2A
fixed_params = {
    'inc': -45.0 * u.deg,  # degrees
    'pa': 194.0 * u.deg,  # degrees
    'mass': 4.0 * u.Msun,  # solar masses
    'rmin': 50.0 * u.au,  # au
    'deltar': 40.0 * u.au,  # au
    'v_lsr': 7.5 * u.km / u.s,  # km/s (systemic velocity)
}

Now we run the forward model for the initial guess, and calculate the loss (i.e. the 'difference' between the model and data streamline)

In [ ]:
# forward model (same inputs as chi2_loss)
model_params, initial_opt_params, fixed_params = gradient_descent.prepare_model_params(initial_opt_params, fixed_params)
ra_model, dec_model, v_model, valid_mask_model, err = gradient_descent.forward_model(
    model_params, distance
)
err.throw()

# match model to data exactly as chi2_loss does
ra_model_interp, dec_model_interp, v_model_interp, valid, model_keep, _dmetric_model, _ = gradient_descent.checked_match_model_to_data_curve(
    ra_model, dec_model, v_model, valid_mask_model, ra_data, dec_data
)

print(initial_opt_params)
print(fixed_params)
print(f"ra_model_interp = {ra_model_interp}")
print(f"ra_model_interp (valid) = {ra_model_interp[valid]}")
print(f"retained {int(jnp.sum(valid))}/{len(valid)} data points after overlap filtering")

# ---- Manual chi2_loss calculation (same logic as gradient_descent.chi2_loss) ----
# Coerce to float64 and floor sigmas to avoid division by zero
ra_data_f = jnp.asarray(ra_data, dtype=jnp.float64)
dec_data_f = jnp.asarray(dec_data, dtype=jnp.float64)
v_data_f = jnp.asarray(v_data, dtype=jnp.float64)

ra_sigma_f = jnp.asarray(ra_sigma, dtype=jnp.float64)
dec_sigma_f = jnp.asarray(dec_sigma, dtype=jnp.float64)
v_sigma_f = jnp.asarray(v_sigma, dtype=jnp.float64)

eps = jnp.asarray(1e-8, dtype=jnp.float64)
ra_sigma_safe = jnp.maximum(ra_sigma_f, eps)
dec_sigma_safe = jnp.maximum(dec_sigma_f, eps)
v_sigma_safe = jnp.maximum(v_sigma_f, eps)

r_data, theta_data = extract_streamline.cartesian_to_polar(ra_data_f, dec_data_f)
r_model, theta_model = extract_streamline.cartesian_to_polar(ra_model_interp, dec_model_interp)

dtheta = extract_streamline.wrap_to_pi(theta_data - theta_model)

sigma_r = jnp.sqrt(ra_sigma**2 + dec_sigma**2)
r_eps = 1e-8
r_safe = jnp.maximum(jnp.abs(r_data), r_eps)
sigma_theta = jnp.sqrt(((dec_data * dec_sigma)**2 + (ra_data * ra_sigma)**2)) / (r_safe**2)
sigma_theta = jnp.maximum(sigma_theta, r_eps)

# Only compute chi2 on valid/retained data points
chi2_r = jnp.sum((((r_data[valid] - r_model[valid]) / sigma_r[valid]) ** 2))
chi2_theta = jnp.sum(((dtheta[valid] / sigma_theta[valid]) ** 2))
chi2_v = jnp.sum((((v_data_f[valid] - v_model_interp[valid]) / v_sigma_safe[valid]) ** 2))
chi2_total = chi2_r + chi2_theta + chi2_v # + chi2_penalty


print(
    f"Chi2 r: {chi2_r:.2f}, Chi2 theta: {chi2_theta:.2f}, "
    f"Chi2 v: {chi2_v:.2f}, Total: {chi2_total:.2f}"
)

# get metric boundaries
partitions = extract_streamline.get_metric_partitions(pc_coords, n_elements=n_points)
metric_boundaries, trace = extract_streamline.sample_metric_boundaries(pc_coords, partitions)
# do the plot using the morphology plotting function
outputs.plot_morphology(
    ra_model=jnp.asarray(ra_model, dtype=jnp.float64),
    dec_model=jnp.asarray(dec_model, dtype=jnp.float64),
    ra_data=jnp.asarray(ra_data, dtype=jnp.float64),
    dec_data=jnp.asarray(dec_data, dtype=jnp.float64),
    ra_sigma=jnp.asarray(ra_sigma, dtype=jnp.float64),
    dec_sigma=jnp.asarray(dec_sigma, dtype=jnp.float64),
    ra_model_interp=jnp.asarray(ra_model_interp, dtype=jnp.float64),
    dec_model_interp=jnp.asarray(dec_model_interp, dtype=jnp.float64),
    valid=jnp.asarray(valid, dtype=bool),
    pc_coords=pc_coords,
    save_folder=save_folder,
    save_name='initial_guess_morphology',
    show=True,
    metric_boundaries=metric_boundaries,
    title=f"Initial Guess Morphology (Loss: {chi2_total:.2f})",
)


Now go back and adjust your initial guess parameters if necessary (e.g. if the parameter combination led to an error because the centrifugal radius is larger than r0).

The initial guess does not need to be good, so do not spend long on this step.

Once happy with your initial guess, continue.

## 3. Set bounds

Each optimisable parameter needs a set of bounds, with units. These are used to a) normalise the parameters so that they each contribute to a similar level in the optimisation, and b) make sure STING does not go off in completely the wrong direction.

It is recommended to define your omega bounds based on your r0 bounds to keep the centrifugal radius (r_c) shorter than r0.

In [ ]:
# def get_omega(mass, r0):
#     '''
#     this gets value of omega when r_cent = 0.5 * r0
#     '''
#     omega_squared = 0.5 * G * mass / (jnp.power(r0, 3) * jnp.power(au_in_km, 2)) # in s^-2
#     omega = jnp.power(omega_squared, 0.5) # in s^-1
#     return omega

opt_params = initial_opt_params.copy()

# Define physically reasonable bounds (omega bounds transformed to natural log space)
# These bounds are also used as normalization anchors: x_norm = (x - min) / (max - min).
# Provide bounds for every optimized parameter.
r0_min, r0_max = 200.0, 10000.0 # param bounds in au
# the omega bounds are set by keeping centrifugal radius reasonable (r_cent = 0.5 r0)
# omega_max = get_omega(fixed_params['mass'], r0_min)
# omega_min = get_omega(fixed_params['mass'], r0_max)
# print these in scientific notation for sanity check
# print(f"Omega bounds: {omega_min:.2e} to {omega_max:.2e} 1/s")

param_bounds = {
    'r0': (r0_min, r0_max) * u.au,                    # radius between 200-20000 au
    'theta0': (0.0, 180.0) * u.deg,                    # polar angle 0-180 degrees
    'phi0': (0.0, 360.0) * u.deg,                    # azimuthal angle 0-360 degrees
    'v_r0': (-5.0, 5.0) * u.km / u.s, # km/s, 
    'omega': (1e-14, 1e-11) * (1/u.s), # 1/s, in log space this is -32.2 to -25.3
    # 'rc': (1.0, r0_max-1e-6) * u.au,  # au, must be less than r0 to avoid singularity
    # 'mu': (0.0, 1.0),  # dimensionless
    # 'mass': (3, 5) * u.Msun,                    # mass between 3.0 and 5.0 solar masses
    # 'inc': (-90.0, 90.0) * u.deg,                    # inclination between -90 and 90 degrees
    # 'pa': (0.0, 360.0) * u.deg,                    # position angle between 0 and 360 degrees
}

## 4. Fit streamline

Choose your settings, or leave them as the defaults, and run STING.

In [ ]:

n_epochs = 300
info_every = 30
learning_rate = 0.002 # Single learning rate applied to all normalized optimization parameters
loss_method = 1 # options: 0: LOSS_RADECVEL, 1: LOSS_RTHETAVEL

gradient_tol = 1e-3 * len(initial_opt_params) # gradient tolerance scaled by number of parameters

## here we run the fit, using cProfile to track performance
profile = False # set to True to enable cProfile profiling of the optimization run
if profile:
    profiler = cProfile.Profile()
    profiler.enable()

jax.config.update("jax_debug_nans", False)
best_opt_params, loss_history, param_errors = gradient_descent.fit_streamline(
    opt_params,
    fixed_params,
    data,
    uncertainties,
    distance,
    learning_rate=learning_rate,
    param_bounds=param_bounds,
    n_epochs=n_epochs,
    info_every=info_every,
    loss_threshold=0.05,
    loss_threshold_epochs=20,
    gradient_tol=gradient_tol,
    gradient_tol_epochs=20,
    early_stopping_patience=210,
    loss_method=loss_method,
    save_folder=save_folder,
    pc_coords=pc_coords,
    v_lsr=v_lsr,
    show_plots=True,
 )

if profile:
    profiler.disable() # Stop profiling after optimization is complete


print(f"Optimized using loss_method='{loss_method}'")

## 5. Optional plots

Some other plots which you might wish to make

### Best fit, overlaid with by-eye fit

In [ ]:
add_by_eye = True
# by eye parameters
by_eye_params = {
    'r0': 2540.0,  # au
    'theta0': 54.0,  # degrees
    'phi0': 61.0,  # degrees
    'log_omega': np.log(7e-13),  # log(1/s)
    'v_r0': 0.001, # km/s
    # 'mass': 4.0,  # solar masses
    # 'inc': -45.0,  # degrees
    # 'pa': 194.0,  # degrees
}
# convert angles to radians for forward model
by_eye_params['theta0'] = np.radians(by_eye_params['theta0'])
by_eye_params['phi0'] = np.radians(by_eye_params['phi0'])
if add_by_eye:
    by_eye_full_params, _, _ = gradient_descent.prepare_model_params(by_eye_params, fixed_params)
    ra_by_eye, dec_by_eye, v_by_eye, valid_mask_by_eye, err = gradient_descent.forward_model(by_eye_full_params, distance)
    err.throw()
    by_eye = (ra_by_eye, dec_by_eye, v_by_eye)

# Final model with best-fit parameters
best_opt_full_params, _, _ = gradient_descent.prepare_model_params(best_opt_params, fixed_params)
ra_best, dec_best, v_best, valid_mask_best, err = gradient_descent.forward_model(best_opt_full_params, distance)
err.throw() 
valid_mask_best = valid_mask_best.astype(bool)
# match to data for plotting
ra_best_interp, dec_best_interp, v_best_interp, valid, model_keep, _dmetric_model, _ = gradient_descent.match_model_to_data_curve(
    ra_best, dec_best, v_best, valid_mask_best, ra_data, dec_data
)

# remove NaN values (due to rmin) from model for plotting
not_nan = ~jnp.isnan(ra_best) & ~jnp.isnan(dec_best) & ~jnp.isnan(v_best)
ra_best = ra_best[not_nan]
dec_best = dec_best[not_nan]
v_best = v_best[not_nan]

outputs.plot_morphology(ra_model=jnp.asarray(ra_best), 
                        dec_model=jnp.asarray(dec_best), 
                        ra_data=jnp.asarray(ra_data), 
                        dec_data=jnp.asarray(dec_data),
                        ra_sigma=jnp.asarray(ra_sigma), 
                        dec_sigma=jnp.asarray(dec_sigma),
                        ra_model_interp=jnp.asarray(ra_best_interp),
                        dec_model_interp=jnp.asarray(dec_best_interp),
                        valid=jnp.asarray(valid, dtype=bool),
                        pc_coords=pc_coords, 
                        by_eye=by_eye, 
                        save_folder=save_folder,
                        save_name='best_fit_morphology',
                        title="Best-fit Morphology",
                        show=True)

outputs.plot_vel_radius(ra_model=jnp.asarray(ra_best), 
                        dec_model=jnp.asarray(dec_best),
                        v_model=jnp.asarray(v_best),
                        ra_data=jnp.asarray(ra_data),
                        dec_data=jnp.asarray(dec_data),
                        v_data=jnp.asarray(v_data),
                        ra_sigma=jnp.asarray(ra_sigma),
                        dec_sigma=jnp.asarray(dec_sigma),
                        v_sigma=jnp.asarray(v_sigma),
                        ra_model_interp=jnp.asarray(ra_best_interp),
                        dec_model_interp=jnp.asarray(dec_best_interp),
                        v_model_interp=jnp.asarray(v_best_interp),
                        valid=jnp.asarray(valid, dtype=bool),
                        pc_coords=pc_coords,
                        by_eye=by_eye,
                        save_folder=save_folder,
                        save_name='best_fit_vel_radius',
                        title="Best-fit Kinematics",
                        show=True)

### Streamline spaghetti
i.e. sample covariance matrix for best-fit parameters and plot `n_samples` possible streamlines

In [ ]:
n_samples = 100
outputs.plot_streamline_covariance_samples(best_opt_params,
                                    initial_opt_params,
                                    fixed_params,
                                    data,
                                    uncertainties,
                                    distance,
                                    param_bounds,
                                    loss_method,
                                    gradient_tol,
                                    v_lsr=v_lsr,
                                    n_samples=n_samples,
                                    save_folder=save_folder)

### Parameter evolution

In [ ]:
outputs.plot_param_optimisation_history(save_folder=save_folder)

### Plots at every epoch

RA - Dec, RA - Velocity, Dec - Velocity, and Velocity - Projected Radius

Set make_video=True to also string each set of these plots into a video

In [ ]:
outputs.plot_morphology_by_epoch(
    param_names=opt_keys,
    gradient_descent=gradient_descent,
    fixed_params=fixed_params,
    distance=distance,
    ra_data=jnp.asarray(ra_data),
    dec_data=jnp.asarray(dec_data),
    ra_sigma=jnp.asarray(ra_sigma),
    dec_sigma=jnp.asarray(dec_sigma),
    pc_coords=jnp.asarray(pc_coords),
    n_points=n_points,
    save_folder=save_folder,
    make_video=True
)

In [ ]:
outputs.plot_ra_vel_by_epoch(
    param_names=opt_keys,
    gradient_descent=gradient_descent,
    fixed_params=fixed_params,
    distance=distance,
    ra_data=ra_data,
    dec_data=dec_data,
    v_data=v_data,
    ra_sigma=ra_sigma,
    v_sigma=v_sigma,
    pc_coords=pc_coords,
    save_folder=save_folder,
    make_video=True
)

In [ ]:
outputs.plot_dec_vel_by_epoch(
    param_names=opt_keys,
    gradient_descent=gradient_descent,
    fixed_params=fixed_params,
    distance=distance,
    ra_data=ra_data,
    dec_data=dec_data,
    v_data=v_data,
    dec_sigma=dec_sigma,
    v_sigma=v_sigma,
    pc_coords=pc_coords,
    save_folder=save_folder,
    make_video=True
)

In [ ]:


# now doing plots of velocity vs projected radius by epoch, with a KDE background
outputs.plot_vel_radius_by_epoch(
    param_names=opt_keys,
    gradient_descent=gradient_descent,
    fixed_params=fixed_params,
    distance=distance,
    ra_data=ra_data,
    dec_data=dec_data,
    v_data=v_data,
    pc_coords=pc_coords,
    ra_sigma=ra_sigma,
    dec_sigma=dec_sigma,
    v_sigma=v_sigma,
    velocity_reference=v_lsr,
    save_folder=save_folder,
    make_video=True
)